# 文書要約のベースラインと ROUGE 評価

書籍 第4章「言語生成問題の解き方」（4-1 文書要約）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、以下の内容を体験します。

1. 要約の評価尺度 **ROUGE-1 / ROUGE-2 / ROUGE-L** の定義式からの実装（4-1-4）
2. 抽出型要約の代表手法（**リード法**・**TextRank**（共有文字バイグラムに基づく）・**LexRank**（文字 n-gram の TF-IDF コサイン類似度に基づく））の実装と比較（4-1-1, 4-1-2）
3. 記事構成（逆ピラミッド型 vs 後半結論型）による手法ごとの得意・不得意の分析
4. 複数文抽出（$k=2$）における冗長性問題と **MMR（Maximal Marginal Relevance）** による多様性の確保（4-1-2）

## 1. 準備とサンプル記事データの作成

日本語の文字単位（文字 n-gram）で評価・抽出を行うため、追加パッケージは不要で、標準ライブラリと `numpy`・`pandas`・`scikit-learn` のみで動作します。

In [1]:
import math
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ニュース記事風の「冒頭に結論がある記事（逆ピラミッド型）」と、解説・ブログ風の「中盤〜後半に結論がある記事」を混ぜた 6 本のサンプル記事（各 5 文）と人手参照要約を用意します。

In [2]:
articles = [
    {
        "id": 1,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "政府は本日、AI分野の研究開発を支援する新たな補助金制度を来年度から開始すると発表しました。",
            "対象となるのは大学やスタートアップ企業で、最大で1件あたり5000万円が支給されます。",
            "近年、生成AIの急速な普及に伴い、国内での計算資源や人材の不足が課題となっていました。",
            "関係省庁は先月から有識者会議を開き、具体的な支援枠組みについて議論を重ねてきました。",
            "申請の受付は来年4月から開始される予定で、詳細は公式ウェブサイトで公開されます。",
        ],
        "reference": "政府は大学やスタートアップ向けに最大5000万円を支給するAI研究開発の新たな補助金制度を来年度から開始します。",
    },
    {
        "id": 2,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "東京都内の主要鉄道各社は、来月1日から終電時刻を約15分繰り上げるダイヤ改正を実施します。",
            "これは深夜の保線作業の時間を確保し、作業員の負担を軽減することが主な目的です。",
            "対象となるのは山手線や中央線などの主要路線で、始発時刻に変更はありません。",
            "利用者の減少傾向も背景にあり、コロナ禍以降、深夜帯の乗車率は約3割低下していました。",
            "各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。",
        ],
        "reference": "都内の主要鉄道各社は深夜の保線作業時間確保のため、来月1日から終電を約15分繰り上げます。",
    },
    {
        "id": 3,
        "type": "ニュース型（冒頭結論）",
        "sentences": [
            "大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。",
            "新開発の電極材料を採用することで、発熱を抑えながら急速充電を実現しています。",
            "同社は2年前から国立研究所と共同で次世代電池の基礎研究を進めてきました。",
            "この技術は電気自動車やスマートフォンへの応用が期待されており、年内の量産開始を目指します。",
            "発表を受けて、本日の株式市場ではA社の株価が一時5%上昇しました。",
        ],
        "reference": "A社が充電時間を半分に短縮した新型バッテリーを開発し、年内の量産開始を目指しています。",
    },
    {
        "id": 4,
        "type": "後半結論型（課題→調査→結論）",
        "sentences": [
            "私たちの街では近年、駅前商店街の空き店舗が増加していることが大きな悩みでした。",
            "以前は休日に多くの家族連れで賑わっていましたが、郊外に大型商業施設ができて状況が一変しました。",
            "市役所は現状を把握するため、住民2000人を対象に買い物行動に関するアンケート調査を実施しました。",
            "調査の結果、若年層は夜間の営業やカフェスペースの充実を求めていることが明らかになりました。",
            "これを受けて市は、空き店舗を改装したコワーキングスペースと夜カフェの誘致を最優先施策として決定しました。",
        ],
        "reference": "住民調査の結果を踏まえ、市は商店街の空き店舗にコワーキングスペースと夜カフェを誘致する施策を決定しました。",
    },
    {
        "id": 5,
        "type": "後半結論型（実験→比較→結論）",
        "sentences": [
            "自然言語処理のモデル開発では、最初から大規模な深層学習モデルを使うべきか議論になることがあります。",
            "大規模モデルは高い精度が期待できる一方、GPUコストや推論速度の面で運用が難しくなる場合があります。",
            "そこで今回、同じ文書分類データに対してルールベース、TF-IDF、BERTの3手法を比較実験しました。",
            "実験では1000件のサンプルデータを用い、正解率と推論時間、誤分類の傾向を詳しく分析しました。",
            "その結果、まずはTF-IDFなどの軽量なベースラインでデータ品質を確認してからBERTへ進むのが最も効率的だとわかりました。",
        ],
        "reference": "比較実験の結果、文書分類ではTF-IDFなどの軽量なベースラインでデータを確認してからBERTへ進むのが最も効率的だとわかりました。",
    },
    {
        "id": 6,
        "type": "後半結論型（背景→問題→結論）",
        "sentences": [
            "リモートワークが定着したことで、オンライン会議の回数が以前より大幅に増えています。",
            "会議の録音や議事録作成の負担を軽減するため、音声認識ツールの導入を検討する企業が増えました。",
            "しかし、専門用語や社内略語が多い会議では、汎用モデルそのままでは誤変換が多発してしまいます。",
            "実際に試したところ、固有名詞の誤認識が原因で議事録の手直しに多くの時間がかかりました。",
            "実用化には、社内の固有表現をユーザー辞書に登録し、さらに要約モデルで要点を整理する仕組みが不可欠です。",
        ],
        "reference": "オンライン会議の議事録自動化を実用化するには、社内固有名詞の辞書登録と要約モデルによる要点整理が不可欠です。",
    },
]
print(f"記事数: {len(articles)} 本")

記事数: 6 本


---

## 2. 評価尺度 ROUGE の実装（4-1-4「自動評価」）

文書要約では、参照要約に含まれる重要な情報（単語・フレーズ）をシステム要約がどれだけ**漏れなく拾えているか（再現率; Recall）**を重視する **ROUGE** が標準的に使われます。

- **ROUGE-1 / ROUGE-2**: 1-gram / 2-gram の重なりに基づく評価（日本語では文字 n-gram または単語 n-gram を用います。ここではトークナイザに依存しない文字 n-gram で計算します）
- **ROUGE-L**: 語順を保った最長共通部分列（Longest Common Subsequence; **LCS**）の長さに基づく評価

ここでは ROUGE-1, ROUGE-2, ROUGE-L の F1 スコアを定義式から実装します。

In [3]:
def char_ngrams(text, n):
    t = text.replace(" ", "").replace("　", "")
    return [t[i:i + n] for i in range(len(t) - n + 1)]

def rouge_n_f1(hyp, ref, n=1):
    h_ng = Counter(char_ngrams(hyp, n))
    r_ng = Counter(char_ngrams(ref, n))
    overlap = sum(min(cnt, r_ng[g]) for g, cnt in h_ng.items())
    h_total = sum(h_ng.values())
    r_total = sum(r_ng.values())
    if overlap == 0 or h_total == 0 or r_total == 0:
        return 0.0
    prec = overlap / h_total
    rec = overlap / r_total
    return 2 * prec * rec / (prec + rec)

def lcs_length(a, b):
    '''動的計画法で文字列 a, b の最長共通部分列（LCS）の長さを求める'''
    m, n = len(a), len(b)
    dp = [0] * (n + 1)
    for i in range(m):
        prev = 0
        for j in range(n):
            temp = dp[j + 1]
            if a[i] == b[j]:
                dp[j + 1] = prev + 1
            else:
                dp[j + 1] = max(dp[j + 1], dp[j])
            prev = temp
    return dp[n]

def rouge_l_f1(hyp, ref):
    h = hyp.replace(" ", "").replace("　", "")
    r = ref.replace(" ", "").replace("　", "")
    if not h or not r:
        return 0.0
    lcs = lcs_length(h, r)
    if lcs == 0:
        return 0.0
    prec = lcs / len(h)
    rec = lcs / len(r)
    return 2 * prec * rec / (prec + rec)

print("ROUGE-1:", round(rouge_n_f1("政府はAIの補助金を開始します", "政府はAI研究開発の補助金を開始します", 1), 3))
print("ROUGE-L:", round(rouge_l_f1("政府はAIの補助金を開始します", "政府はAI研究開発の補助金を開始します"), 3))

ROUGE-1: 0.882
ROUGE-L: 0.882


---

## 3. 抽出型要約の 3 手法を実装・比較する（4-1-1, 4-1-2）

記事の 5 文から **要約として 1 文** を抽出するタスクとして、書籍 4-1-1 / 4-1-2（p.131）で解説されている次の 3 手法を比較します。

1. **リード法（Lead）**: 常に先頭の 1 文目を選ぶ単純なベースライン
2. **TextRank**: 2 文間の共通要素数に基づく類似度 $\mathrm{Sim}(S_i, S_j) = \frac{|\{w_k \mid w_k \in S_i \ \& \ w_k \in S_j\}|}{\log(|S_i|) + \log(|S_j|)}$ を用いて PageRank（べき乗法）で中心文を選ぶ
   この演習の TextRank では、各文を重複を除いた文字バイグラムの集合として表し、式の分母に使う各集合の要素数は最低2とします。
3. **LexRank**: TF-IDF（文字 2〜3-gram）のコサイン類似度行列を用いて PageRank（べき乗法）で中心文を選ぶ

In [4]:
def pagerank_select(sim, sentences, k=1, damping=0.85, n_iter=30):
    np.fill_diagonal(sim, 0.0)
    row_sums = sim.sum(axis=1, keepdims=True)
    trans = np.divide(sim, row_sums, out=np.full_like(sim, 1.0 / len(sentences)), where=row_sums != 0)
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(n_iter):
        scores = (1.0 - damping) / len(sentences) + damping * trans.T @ scores
    top_idx = sorted(np.argsort(scores)[::-1][:k])
    return "".join(sentences[i] for i in top_idx)

def summarize_lead(sentences, k=1):
    return "".join(sentences[:k])

def summarize_textrank(sentences, k=1):
    # 書籍 p.131 の TextRank 類似度（共通バイグラム数 / (log|Si| + log|Sj|)）
    sets = [set(char_ngrams(s, 2)) for s in sentences]
    n = len(sentences)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                denom = math.log(max(len(sets[i]), 2)) + math.log(max(len(sets[j]), 2))
                sim[i, j] = len(sets[i] & sets[j]) / denom
    return pagerank_select(sim, sentences, k=k)

def summarize_lexrank(sentences, k=1):
    # 書籍 p.131 の LexRank 類似度（TF-IDF のコサイン類似度）
    vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    mat = vec.fit_transform(sentences)
    sim = cosine_similarity(mat)
    return pagerank_select(sim, sentences, k=k)

methods = {
    "リード法": summarize_lead,
    "TextRank": summarize_textrank,
    "LexRank": summarize_lexrank,
}

eval_rows = []
for art in articles:
    for m_name, fn in methods.items():
        hyp = fn(art["sentences"], k=1)
        eval_rows.append({
            "記事ID": art["id"],
            "構成タイプ": art["type"],
            "手法": m_name,
            "ROUGE-1": round(rouge_n_f1(hyp, art["reference"], 1), 3),
            "ROUGE-2": round(rouge_n_f1(hyp, art["reference"], 2), 3),
            "ROUGE-L": round(rouge_l_f1(hyp, art["reference"]), 3),
            "抽出された文": hyp,
        })

res_df = pd.DataFrame(eval_rows)
print("=== 全記事の平均 ROUGE スコア ===")
display(res_df.groupby("手法")[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean().round(3))

print("\n=== 記事の構成タイプ別の平均 ROUGE-L ===")
display(res_df.pivot_table(index="構成タイプ", columns="手法", values="ROUGE-L", aggfunc="mean").round(3))

=== 全記事の平均 ROUGE スコア ===


,ROUGE-1,ROUGE-2,ROUGE-L
手法,,,
LexRank,0.449,0.255,0.340
TextRank,0.563,0.393,0.489
リード法,0.515,0.343,0.431



=== 記事の構成タイプ別の平均 ROUGE-L ===


手法,LexRank,TextRank,リード法
構成タイプ,,,
ニュース型（冒頭結論）,0.351,0.465,0.620
後半結論型（実験→比較→結論）,0.342,0.891,0.191
後半結論型（背景→問題→結論）,0.280,0.280,0.232
後半結論型（課題→調査→結論）,0.367,0.367,0.304


### 記事ごとの抽出結果を確認する

スコアだけでなく、実際にどの文が選ばれたかを参照要約と見比べてみましょう。

In [5]:
for art in articles:
    print(f"[記事 {art['id']}: {art['type']}]")
    print(f"  参照要約: {art['reference']}")
    sub = res_df[res_df["記事ID"] == art["id"]]
    for _, row in sub.iterrows():
        print(f"  {row['手法']:8s} (R-L={row['ROUGE-L']:.3f}): {row['抽出された文']}")
    print()

[記事 1: ニュース型（冒頭結論）]
  参照要約: 政府は大学やスタートアップ向けに最大5000万円を支給するAI研究開発の新たな補助金制度を来年度から開始します。
  リード法     (R-L=0.549): 政府は本日、AI分野の研究開発を支援する新たな補助金制度を来年度から開始すると発表しました。
  TextRank (R-L=0.549): 政府は本日、AI分野の研究開発を支援する新たな補助金制度を来年度から開始すると発表しました。
  LexRank  (R-L=0.208): 申請の受付は来年4月から開始される予定で、詳細は公式ウェブサイトで公開されます。

[記事 2: ニュース型（冒頭結論）]
  参照要約: 都内の主要鉄道各社は深夜の保線作業時間確保のため、来月1日から終電を約15分繰り上げます。
  リード法     (R-L=0.689): 東京都内の主要鉄道各社は、来月1日から終電時刻を約15分繰り上げるダイヤ改正を実施します。
  TextRank (R-L=0.225): 各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。
  LexRank  (R-L=0.225): 各社は駅構内のポスターやアプリを通じて、利用者への周知を進めています。

[記事 3: ニュース型（冒頭結論）]
  参照要約: A社が充電時間を半分に短縮した新型バッテリーを開発し、年内の量産開始を目指しています。
  リード法     (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。
  TextRank (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。
  LexRank  (R-L=0.621): 大手電機メーカーのA社は、従来比で充電時間を半分に短縮した新型バッテリーを開発しました。

[記事 4: 後半結論型（課題→調査→結論）]
  参照要約: 住民調査の結果を踏まえ、市は商店街の空き店舗にコワーキングスペースと夜カフェを誘致する施策を決定しました。
  リード法     (R-L=0.304): 私たちの街では近年、駅前商店街の空き店舗が増加していることが大きな悩みでした。
  TextRank (

### 観察とまとめ

1. **ニュース型（冒頭結論）ではリード法が非常に強い**: 新聞・ニュース記事は最初の文に要点をまとめる慣習（逆ピラミッド構造）があるため、単純なリード法が強力なベースラインになります（書籍 4-1-1）。
2. **後半結論型では内容ベースの手法が有利**: 課題や実験の経緯を述べてから最後に結論を書く構成では、リード法は導入文を選んでしまいますが、TextRank や LexRank は記事全体のキーワードが集約された文を拾えることがあります。
3. **抽出型要約の限界**: 抽出型要約は文をそのままコピーするため文法的な破綻は起きにくい一方、参照要約のように複数の文の要素を 1 文に圧縮・言い換えすること（**抽象型要約**）はできません。

---

## 4. 複数文抽出（$k=2$）の冗長性問題と MMR（4-1-2）

第 3 節では 1 文だけを抽出しましたが、実際の要約では複数の文（例：$k=2$）を抜き出すことがよくあります。このとき、記事内に**同じ見出し内容を少し言い換えて繰り返す文**（例：文1 と 文2）が含まれると、互いに高い類似度を持つため両方の重要度スコアが高くなり、通常の LexRank（上位 $k$ 文選択）は**内容が重複した冗長な 2 文（文1 + 文2）**を選んでしまいます。

この冗長性を抑える手法が **MMR（Maximal Marginal Relevance）** です。各文 $s_i$ の重要度 $\mathrm{Rel}(s_i)$（ここでは最大値で 0〜1 に正規化した LexRank スコア）から、すでに選択した文集合 $S$ との最大類似度（文字 2〜3-gram TF-IDF コサイン類似度）をペナルティとして差し引きながら 1 文ずつ貪欲に選びます。

$$\mathrm{MMR}(s_i) = \lambda \cdot \mathrm{Rel}(s_i) - (1 - \lambda) \max_{s_j \in S} \mathrm{Sim}(s_i, s_j)$$

ここでは $\lambda = 0.6$ とし（$\lambda = 1.0$ のときは通常の LexRank と一致）、見出しの言い換えが重複する 3 本のニュース記事（各 6 文、2 文の参照要約付き）で通常の LexRank（$k=2$）と MMR（$k=2$）を比較します。

In [6]:
mmr_articles = [
    {
        "id": 1,
        "title": "自治体DX補助金制度",
        "sentences": [
            "総務省は本日、全国の自治体で行政手続きのオンライン化と窓口業務のデジタル化を支援する新たな補助金制度を来年度から開始すると発表しました。",
            "来年度から開始される総務省の新たな補助金制度は、全国の自治体における窓口業務のデジタル化と行政手続きのオンライン化を支援するものです。",
            "これまで各地では申請書類の紙処理や窓口対応の負担が課題で、予算措置による早期の事務効率化が求められていました。",
            "政府の検討会議でも、来月15日からの申請受付や1件あたり最大3000万円の上限額など交付基準の枠組みを詰めてきました。",
            "今回の予算総額は120億円で、来月15日から申請を受け付け、採択自治体へ1件あたり最大3000万円を交付する計画です。",
            "なお、採択自治体の公表や交付金の詳しい申請方法については、来週公開される特設サイトで確認できます。",
        ],
        "reference": "総務省は全国の自治体で手続きオンライン化と窓口のデジタル化を支援する新補助金制度を来年度から始めます。総予算120億円で来月15日から申請を受け付け、1件最大3000万円を交付します。",
    },
    {
        "id": 2,
        "title": "自動運転バス実証実験",
        "sentences": [
            "国土交通省と北九州市は本日、市内中心部の幹線道路で自動運転バスの定期運行を行う実証実験を今年秋から開始すると発表しました。",
            "今年秋から開始される実証実験は、国土交通省と北九州市が市内中心部の幹線道路で自動運転バスの定期運行を行うものです。",
            "地方都市では路線バスの運転手不足が進んでおり、利用しやすい予約導線と安定した運行ダイヤの維持が課題となっていました。",
            "運行開始に向けて、導入車両には周囲の障害物を検知する最新センサーを搭載し、専用アプリによる事前予約や安全確認テストを重ねています。",
            "車両には最新センサーを搭載し、利用者は専用アプリから無料で予約でき、1日あたり12便を運行する予定です。",
            "乗客が専用アプリで無料予約できる1日12便の運行データは来春までに集計され、今後の本格展開に活用されます。",
        ],
        "reference": "国交省と北九州市は今年秋から市内幹線道路で自動運転バスの定期運行実験を始めます。最新センサー搭載車両を用い、専用アプリから無料予約できて1日12便を運行する予定です。",
    },
    {
        "id": 3,
        "title": "全固体電池の量産技術開発",
        "sentences": [
            "自動車大手のA社は本日、電気自動車向けに充電時間を従来の半分へ短縮する新型全固体電池の量産技術を確立したと発表しました。",
            "A社が確立した新たな量産技術は、電気自動車向けの新型全固体電池において充電時間を従来の半分へ短縮するものです。",
            "従来の車載電池は急速充電時の発熱が障壁でしたが、材料改良によって高い耐久性と生産効率を両立させました。",
            "すでに栃木工場での試作ライン検証を終えており、月産1万台規模の生産体制と800キロメートル走行の目標達成にめどを付けました。",
            "来年春から栃木工場で月産1万台分の生産を始め、1回の充電で800キロメートル走れる新型車へ搭載する計画です。",
            "なお、新型車の販売価格帯や予約受付の時期については、来月開かれる事業説明会で公表する予定です。",
        ],
        "reference": "A社は充電時間を半減するEV向け新型全固体電池の量産技術を確立しました。来春から栃木工場で月産1万台を生産し、1充電で800キロ走れる新型車へ搭載します。",
    },
]

def compute_lexrank(sentences, damping=0.85, n_iter=30):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    mat = vec.fit_transform(sentences)
    sim = cosine_similarity(mat)
    sim_pr = sim.copy()
    np.fill_diagonal(sim_pr, 0.0)
    row_sums = sim_pr.sum(axis=1, keepdims=True)
    trans = np.divide(sim_pr, row_sums, out=np.full_like(sim_pr, 1.0 / len(sentences)), where=row_sums != 0)
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(n_iter):
        scores = (1.0 - damping) / len(sentences) + damping * trans.T @ scores
    return scores, sim

def select_mmr(sentences, k=2, lam=0.6):
    scores, sim = compute_lexrank(sentences)
    rel = scores / scores.max()
    selected = []
    remaining = list(range(len(sentences)))
    while len(selected) < k and remaining:
        if not selected:
            best = max(remaining, key=lambda i: rel[i])
        else:
            best = max(
                remaining,
                key=lambda i: lam * rel[i] - (1.0 - lam) * max(sim[i, j] for j in selected),
            )
        selected.append(best)
        remaining.remove(best)
    idx = sorted(selected)
    return idx, "".join(sentences[i] for i in idx), rel, sim

# 記事 1 について 2 文目選択時の MMR スコア内訳を確認
art1 = mmr_articles[0]
_, _, rel1, sim1 = select_mmr(art1["sentences"], k=2, lam=0.6)
first1 = int(np.argmax(rel1))
breakdown_rows = []
for i, sent in enumerate(art1["sentences"]):
    mmr_s = None if i == first1 else round(float(0.6 * rel1[i] - 0.4 * sim1[i, first1]), 3)
    breakdown_rows.append({
        "文番号": f"文{i + 1}",
        "正規化 LexRank 中心性 Rel": round(float(rel1[i]), 3),
        f"文{first1 + 1}との類似度 Sim": round(float(sim1[i, first1]), 3),
        "2文目 MMR スコア (λ=0.6)": "選択済み(1文目)" if mmr_s is None else mmr_s,
        "文（冒頭）": sent[:32] + "…",
    })
print("=== 記事 1（自治体DX補助金制度）の 2 文目選択スコア内訳 ===")
display(pd.DataFrame(breakdown_rows))

mmr_rows = []
for art in mmr_articles:
    idx_lr, hyp_lr, _, _ = select_mmr(art["sentences"], k=2, lam=1.0)
    idx_mmr, hyp_mmr, _, _ = select_mmr(art["sentences"], k=2, lam=0.6)
    for m_name, idx, hyp in [
        ("LexRank (k=2)", idx_lr, hyp_lr),
        ("MMR (λ=0.6, k=2)", idx_mmr, hyp_mmr),
    ]:
        mmr_rows.append({
            "記事": f"{art['id']}. {art['title']}",
            "手法": m_name,
            "選択された文": "+".join(f"文{i + 1}" for i in idx),
            "ROUGE-1": round(rouge_n_f1(hyp, art["reference"], 1), 3),
            "ROUGE-2": round(rouge_n_f1(hyp, art["reference"], 2), 3),
            "ROUGE-L": round(rouge_l_f1(hyp, art["reference"]), 3),
        })

mmr_df = pd.DataFrame(mmr_rows)
print("\n=== 記事別の複数文抽出 (k=2) 比較 ===")
display(mmr_df)

print("\n=== 平均 ROUGE スコア（3記事平均） ===")
display(mmr_df.groupby("手法")[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean().round(3))

print("\n=== 記事 1 の出力例比較 ===")
print("参照要約         :", art1["reference"])
print("LexRank (k=2)    :", select_mmr(art1["sentences"], k=2, lam=1.0)[1])
print("MMR (λ=0.6, k=2) :", select_mmr(art1["sentences"], k=2, lam=0.6)[1])

=== 記事 1（自治体DX補助金制度）の 2 文目選択スコア内訳 ===


,文番号,正規化 LexRank 中心性 Rel,文1との類似度 Sim,2文目 MMR スコア (λ=0.6),文（冒頭）
0,文1,1.000,1.000,選択済み(1文目),総務省は本日、全国の自治体で行政手続きのオンライン化と窓口業務の…
1,文2,0.996,0.567,0.37,来年度から開始される総務省の新たな補助金制度は、全国の自治体にお…
2,文3,0.223,0.028,0.122,これまで各地では申請書類の紙処理や窓口対応の負担が課題で、予算措…
3,文4,0.670,0.027,0.391,政府の検討会議でも、来月15日からの申請受付や1件あたり最大30…
4,文5,0.795,0.024,0.468,今回の予算総額は120億円で、来月15日から申請を受け付け、採択…
5,文6,0.339,0.011,0.199,なお、採択自治体の公表や交付金の詳しい申請方法については、来週公…



=== 記事別の複数文抽出 (k=2) 比較 ===


,記事,手法,選択された文,ROUGE-1,ROUGE-2,ROUGE-L
0,1. 自治体DX補助金制度,LexRank (k=2),文1+文2,0.546,0.382,0.493
1,1. 自治体DX補助金制度,"MMR (λ=0.6, k=2)",文1+文5,0.822,0.673,0.795
2,2. 自動運転バス実証実験,LexRank (k=2),文1+文2,0.527,0.332,0.408
3,2. 自動運転バス実証実験,"MMR (λ=0.6, k=2)",文1+文5,0.816,0.608,0.714
4,3. 全固体電池の量産技術開発,LexRank (k=2),文1+文2,0.510,0.326,0.406
5,3. 全固体電池の量産技術開発,"MMR (λ=0.6, k=2)",文1+文5,0.754,0.614,0.712



=== 平均 ROUGE スコア（3記事平均） ===


,ROUGE-1,ROUGE-2,ROUGE-L
手法,,,
LexRank (k=2),0.528,0.347,0.436
"MMR (λ=0.6, k=2)",0.797,0.632,0.740



=== 記事 1 の出力例比較 ===
参照要約         : 総務省は全国の自治体で手続きオンライン化と窓口のデジタル化を支援する新補助金制度を来年度から始めます。総予算120億円で来月15日から申請を受け付け、1件最大3000万円を交付します。
LexRank (k=2)    : 総務省は本日、全国の自治体で行政手続きのオンライン化と窓口業務のデジタル化を支援する新たな補助金制度を来年度から開始すると発表しました。来年度から開始される総務省の新たな補助金制度は、全国の自治体における窓口業務のデジタル化と行政手続きのオンライン化を支援するものです。
MMR (λ=0.6, k=2) : 総務省は本日、全国の自治体で行政手続きのオンライン化と窓口業務のデジタル化を支援する新たな補助金制度を来年度から開始すると発表しました。今回の予算総額は120億円で、来月15日から申請を受け付け、採択自治体へ1件あたり最大3000万円を交付する計画です。


3 本すべての記事で次の現象が確認できます。

- **通常の LexRank（$k=2$）**: 見出しを言い換えた文 1 と文 2 が互いの中心性を押し上げるため、**内容が重複する `文1+文2`** を選んでしまい、予算規模・実施時刻・生産台数といった具体的な補足情報を取りこぼします。
- **MMR（$\lambda = 0.6, k=2$）**: 1 文目に文 1 を選んだあと、文 1 とのコサイン類似度が高い文 2 には大きな冗長性ペナルティ（記事 1 では $\mathrm{Sim}=0.567$）がかかるため、類似度が低く中心性も高い具体的な事実の文（**`文1+文5`**）が選ばれます。その結果、3 記事すべてで ROUGE-1 / ROUGE-2 / ROUGE-L が大きく向上しています。